# Running photoD on Rubin DP2

A clone of this branch has the locus in `data/`. The TRILEGAL prior maps are too large for the repository and
are passed with `--priors`; `scripts/make_priors.py` builds them.

    python scripts/run_dp2.py --catalog <rubin_dp2/object_collection> --priors <priors.npz> --out <dir> --name dp2_photod --workers 6

This notebook does that on one field first, checks the answer, and then runs the survey.

## 1. What a clone gives you

In [ ]:
import glob, subprocess, sys
from pathlib import Path

from photod.locus import RUN_LOCUS_FILE   # the locus a run fits against

photod  = Path("..").resolve()          # the checkout
catalog = "/astro/store/shire/hats/catalogs/rubin_dp2/object_collection"
priors  = "/path/to/priors.npz"         # the TRILEGAL prior maps
out     = "/tmp/photod"                 # where the answers go

for name, f in (("locus", RUN_LOCUS_FILE), ("prior maps", Path(priors))):
    print(f"{name:12s} {'ok' if f.exists() else 'MISSING':>8}  {f.stat().st_size / 2**20 if f.exists() else 0:7.1f} MB  {f}")

## 2. One field first

Always worth doing before committing to the survey: it takes a couple of minutes and catches a wrong path or
a missing column straight away. `--cone RA DEC RADIUS` takes a piece of sky.

In [ ]:
%%time
def run(*extra):
    command = [sys.executable, str(photod / "scripts/run_dp2.py"),
               "--catalog", catalog, "--priors", priors, "--out", out, *extra]
    print(" ".join(command[2:]), "\n")
    done = subprocess.run(command, text=True)
    assert done.returncode == 0, f"the run failed with {done.returncode}"

run("--name", "field", "--cone", "287.33", "-22.985", "0.4", "--workers", "4", "--overwrite")

## 3. Did every star come out the other side

The check worth keeping: the fit should answer for every star the selection produced. It is the one thing
that catches a step quietly dropping rows, which is easy to miss because the stars that survive look fine.

A star the fit cannot be run on at all keeps its row too, with empty estimates and a flag saying why: no
prior map covers its part of the sky (bit 64), or it has no r magnitude (bit 32). The prior maps are built
for a footprint, and a footprint never lines up exactly with the survey that produced the stars.

In [ ]:
import pyarrow.parquet as pq

def rows(path):
    return sum(pq.ParquetFile(f).metadata.num_rows
               for f in glob.glob(f"{path}/dataset/**/*.parquet", recursive=True))

print(f"{rows(f'{out}/field'):,} stars fitted")

## 4. What the fit gives you

Per star: the 16th, 50th and 84th percentiles and the standard deviation of the absolute magnitude `Mr`, the
position along the locus `tLoc`, the metallicity `FeH`, the extinction `Ar` and the distance modulus `DM`, then
`multimodality`, `chi2min` and `fit_flags`. `Mr` is the absolute magnitude; `tLoc` equals it only on the main
sequence fainter than the turn-off (`tLoc` > 4), so plot `Mr`.

The distance is `10 ** (DM / 5 + 1)` parsecs. `DM` comes from the posterior of `Mr + Ar` rather than from
combining two marginals, so its percentiles are the right ones.

How good an answer is lives in columns, not in flags:

- `chi2min`, the chi-squared of the best fit with five colours (fewer when some are missing);
- `multimodality`, how much of the distance-modulus posterior lies in a second peak, measured after smoothing
  it by 0.2 mag so that the separate [Fe/H] tracks of the giant branch do not count as peaks;
- `<name>_nongaussianity` for `Mr`, `FeH`, `Ar` and `DM`, the standard deviation over half the 16-84 %
  interval: one for a Gaussian, larger with heavy tails or a second peak;
- `<name>_asymmetry`, the length of the interval above the median over the length below it: one when symmetric.

For a cone or a small catalog, `--save-posteriors` also writes each star's one-dimensional posteriors (about
26 kB a star) to `<out>/<name>/posteriors`, to look at stars closely.

In [ ]:
import numpy as np

def readAll(path, columns):
    files = sorted(glob.glob(f"{path}/dataset/**/*.parquet", recursive=True))
    step = max(1, len(files) // 300)
    got = {c: [] for c in columns}
    for f in files[::step]:
        t = pq.read_table(f, columns=columns)
        for c in columns:
            got[c].append(t[c].to_numpy(zero_copy_only=False))
    return {c: np.concatenate(v) for c, v in got.items()}

cols = ["rmag", "chi2min", "multimodality", "fit_flags", "DM_quantile_median", "Mr_quantile_median",
        "FeH_quantile_median", "Ar_quantile_median"]
d = readAll(f"{out}/field", cols)
for c in cols:
    v = d[c].astype(float)
    print(f"  {c:22s} 5/50/95 %: " + "  ".join(f"{x:8.3f}" for x in np.nanpercentile(v, [5, 50, 95])))

### fit_flags

One bit per fact about the row: the fit was not run, an input was missing, or a number is a limit of the
grid rather than a measurement. None of them says the answer is bad, and nothing is dropped for being
flagged.

| bit | meaning |
|---|---|
| 4 | [Fe/H] is against the metal-poor end of the model grid, so it is a limit rather than a measurement |
| 8 | A_r is against the top of what the grid and the dust map allow, and the distance goes wrong with it |
| 16 | at least one colour was not measured and carried no weight (which one: bits 16384 to 262144) |
| 32 | no r magnitude, so neither a prior map nor a distance: the row carries no estimate |
| 64 | no prior map for this part of the sky, so the fit was never run: no estimate either |
| 128 | the posterior fell into a single grid cell: precise-looking and not to be trusted |
| 8192 | [Fe/H] is against the metal-rich end of the model grid |
| 16384, 32768, 65536, 131072, 262144 | u-g, g-r, r-i, i-z, z-y was not measured |

Bits 1 and 2 (a poor fit at chi2 above 100, a lopsided `Mr` posterior) were set by earlier versions and are
no longer. A missing colour is normal rather than a fault: it is nearly all u-g, the shallowest, and about
half of DP2 has none.

Where to cut is the science's choice. Two numbers to start from. Against Gaia parallaxes, in the October
DP2 run, the stars with `chi2min` above 100 scatter five to twenty five times their quoted uncertainty where
the rest scatter 1.09 times it. On a TRILEGAL test where 0.75 % of the stars land on the wrong branch (more
than 1.7 mag from their true `Mr`), `multimodality > 0.05` catches 81 % of those and 7 % of the rest, and
`> 0.1` catches 71 % and 4 %.

In [ ]:
flags = d["fit_flags"].astype(int)
fitted = np.isfinite(d["DM_quantile_median"].astype(float))
print(f"{len(flags):,} stars, {100 * fitted.mean():.2f} % with an estimate")
for bit, label in ((4, "FeH at the metal-poor end"), (8192, "FeH at the metal-rich end"),
                   (8, "A_r at the grid top"), (16, "a colour missing"), (32, "no r magnitude"),
                   (64, "no prior map for that sky"), (128, "collapsed")):
    print(f"  bit {bit:6d} {label:26s}: {100 * np.mean((flags & bit) > 0):6.2f} %")
for k, colour in enumerate(("u-g", "g-r", "r-i", "i-z", "z-y")):
    print(f"  bit {16384 << k:6d} {colour + ' missing':26s}: {100 * np.mean((flags & (16384 << k)) > 0):6.2f} %")
chi2, mm = d["chi2min"].astype(float), d["multimodality"].astype(float)
print(f"of those with an estimate: chi2min > 100 {100 * np.mean(chi2[fitted] > 100):.2f} %, "
      f"multimodality > 0.05 {100 * np.mean(mm[fitted] > 0.05):.2f} %")

## 5. Does it look like the Galaxy

The cheapest check that the distances and the metallicities mean anything: take stars bright enough in
absolute terms to be seen across the whole range, so the sample is volume limited, and look at the
metallicity against height above the plane. It should run from the thin disc near the plane to the halo
several kiloparsecs up, and nothing in the fit was told to do that.

In [ ]:
from astropy.coordinates import SkyCoord

p = readAll(f"{out}/field", ["ra", "dec", "fit_flags", "chi2min", "multimodality", "DM_quantile_median",
                             "Mr_quantile_median", "FeH_quantile_median"])
# one choice of cut for this check, not the cut: fitted, not collapsed, a good fit, one branch
keep = np.isfinite(p["DM_quantile_median"].astype(float)) & (p["fit_flags"].astype(int) & 128 == 0)
keep &= (p["chi2min"].astype(float) < 100) & (p["multimodality"].astype(float) < 0.05)
keep &= (p["Mr_quantile_median"].astype(float) > 4) & (p["Mr_quantile_median"].astype(float) < 6)
b = SkyCoord(p["ra"][keep], p["dec"][keep], unit="deg").galactic.b.deg
z = np.abs(10 ** (p["DM_quantile_median"].astype(float)[keep] / 5 + 1) * np.sin(np.radians(b))) / 1000
feh = p["FeH_quantile_median"].astype(float)[keep]
print(f"{'|z| kpc':>12} {'stars':>8} {'[Fe/H] median':>15}")
for lo, hi in ((0.3, 1), (1, 2), (2, 4), (4, 9)):
    s = (z >= lo) & (z < hi)
    if s.sum() > 30:
        print(f"{lo:5.1f}-{hi:<6.1f} {int(s.sum()):8d} {np.median(feh[s]):15.3f}")

## 6. The whole survey

Same command without `--cone`. On two L40s it took 93 minutes for 93.7 million stars, about 16800 a second
end to end, and wrote 16 GB. `--workers` is processes, which share whatever GPUs are there; two per card is
a reasonable start. Memory stays flat because a worker is replaced every few hundred partitions.

A run that stops part way is finished by repeating the command without `--overwrite`: the partitions already
written are kept and only the rest are fitted. A partition that fails is reported and counted, the run goes
on to the end, and its exit status is non-zero so that repeating the command is the obvious next step.

Uncomment and run when you are ready.

In [ ]:
# run("--name", "dp2_photod", "--workers", "6", "--overwrite")

## 7. If your catalog is already prepared

A catalog that already carries `ra`, `dec`, `rmag`, the five colours and their errors is used as it is; the
fit only builds colours from PSF fluxes when it has to. Name the extinction column it came with, and the factor that puts it on this
code's A_r/E(B-V) of 2.323 (Lovorka's `Ar_SFD` is 2.701 × ebv):

    --ar-column Ar_SFD --ar-scale 0.86005183

## 8. Another footprint, or the exact maps

`scripts/make_priors.py` builds prior maps for a footprint out of TRILEGAL, `--compact` for the small form,
and `--from-catalog` converts maps that already exist as a HATS catalog.
The 3D dust prior is off and its curves no longer ship here; `--dust-curves` takes such a file if one is built elsewhere.

The compact form moves the median star by under two millimagnitudes against the full-resolution maps.